# HYPER-3, 4 — The dose–response curve, and which doses phase III should test

Four arms give four points. A phase III has to choose doses that were never tested, in
a population weighted differently from this one, and it gets one attempt. That is a
question about a *curve*, not about four means, so this notebook fits one.

`axiom.surface` is the response-surface layer: a `SurfaceSpec` names a saturating
kernel per treatment, `build` turns it into the single `ModelSpec` whose mean the
likelihood, the simulator, the design math and the optimizer all evaluate, and `fit`
runs a backend. `axiom.design`'s structural half then asks the design question — where
to place the next doses so the parameters are actually identified.

It also produces the notebook's most useful negative result — and then the fix. The
*saturating* kernels are monotone by construction: more dose, more effect, levelling
off. The oldest stratum's curve turns around, and a Hill fit cannot describe that; it
says so with a residual rather than with a warning. Section 3b swaps in the **basis**
families (polynomial, natural cubic spline, piecewise linear), which can.

In [ ]:
import numpy as np
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots

import hyper3 as h
from axiom.core import BASES, D, Outcome, Treatment, Unsupported, is_failure
from axiom.data import Panel, RoleMap
from axiom.design import (
    FisherInformation, IdentifiabilityRidge, IdentifyingDesign, design_to_identify,
    expected_posterior_sd, fisher_information, identifiability_ridge, ridge_of,
)
from axiom.surface import (
    AnyKernel, Bounds, Design, FitResult, GaussianProcessKernel, HillKernel,
    PiecewiseLinearKernel, PolynomialKernel, SplineKernel, Surface, SurfaceSpec, build,
    equal_spacing, fit, forward, full_factorial, marginal, parameter_roles, predict,
)
from axiom.viz import available, response_curve

from axiom.display import enable

enable();  # every axiom result renders itself from here on

BASES.declare("mass", symbol="M")
SEED = 20260821
trial = h.trial(seed=SEED)
final = h.look_frame(trial, h.TRIAL_WEEKS, endpoint="primary")

reduction = Outcome(name="reduction", dimension=D.outcome, unit=h.OUTCOME_UNIT, aggregation="mean")
drug = Treatment(name="dose", dimension=D.mass, unit=h.DOSE_UNIT)
spec = SurfaceSpec(
    name="hyper3_dose_response", treatments=(drug,), outcome=reduction,
    kernels={"dose": HillKernel(reference_dose=20.0, amplitude_scale=8.0)},
    intercept="shared", noise_scale=6.0,
)
model = build(spec)
print("model parameters:", [p.name for p in model.parameters])
print("roles            :", parameter_roles(spec))
print("mean expression  :", model.mean.node)
print("\nthe outcome is signed so that *positive is better*: a reduction in pressure.")

## 1. One surface per stratum

Nothing in the kernel knows about age, so the stratification is done the blunt way:
three fits on three disjoint sets of units, each with its own intercept, amplitude,
half-maximal dose and shape. Each fit sees one period — the primary window — so this is
the designed-experiment special case of the panel surface, with no carryover.

In [ ]:
def stratum_panel(stratum: str) -> Panel:
    rows = final[final["stratum"] == stratum].copy()
    rows["reduction"] = -rows["change"]
    rows["t"] = 0
    return Panel(rows[["unit", "t", "reduction", "dose"]],
                 RoleMap(unit="unit", time="t", outcome=("reduction", reduction),
                         treatments={"dose": drug}))


fits: dict[str, FitResult] = {}
for stratum in h.STRATA:
    fits[stratum] = fit(spec, stratum_panel(stratum), backend="laplace", draws=800,
                        chains=1, seed=SEED)
    result = fits[stratum]
    assert result.converged, stratum
    row = "  ".join(f"{name}={result.posterior.summary(name).mean:6.2f}"
                    for name in ("alpha", "beta_dose", "k_dose", "s_dose", "sigma"))
    print(f"{h.STRATUM_LABEL[stratum]:>6s}  {row}")

`beta_dose` is the maximum reduction the curve can reach, `k_dose` the dose that gets
halfway there, `s_dose` the steepness. Read them against the truth: the generating Emax
is 9.5 / 11.0 / 12.5 mmHg at a half-maximal dose of 18 mg, *before* adherence, and the
fitted amplitudes are lower because the estimand is intent to treat.

## 2. The curves, against the arm means they came from

In [ ]:
grid = np.linspace(0.0, 48.0, 97)
observed = h.mean_with_error(
    final.assign(reduction=-final["change"]), ["stratum", "dose"], "reduction")
fig = make_subplots(rows=1, cols=3, shared_yaxes=True,
                    subplot_titles=[h.STRATUM_LABEL[s] for s in h.STRATA])
for col, stratum in enumerate(h.STRATA, start=1):
    result = fits[stratum]
    draws = predict(result.surface, result.posterior, {"dose": grid}, seed=SEED)
    curve = np.asarray(draws.values).reshape(-1, grid.size)
    lower, mid, upper = np.percentile(curve, [5, 50, 95], axis=0)
    fig.add_trace(go.Scatter(x=list(grid) + list(grid)[::-1], y=list(upper) + list(lower)[::-1],
                             fill="toself", fillcolor=h._rgba(h.STRATUM_COLOR[stratum], 0.18),
                             line={"width": 0}, showlegend=False, hoverinfo="skip"),
                  row=1, col=col)
    fig.add_trace(go.Scatter(x=grid, y=mid, mode="lines", name="fitted Hill curve",
                             line={"color": h.STRATUM_COLOR[stratum], "width": 2.6},
                             showlegend=col == 1), row=1, col=col)
    arm = observed[observed["stratum"] == stratum]
    fig.add_trace(go.Scatter(x=arm["dose"], y=arm["mean"], mode="markers", name="arm mean",
                             marker={"color": "#111", "size": 9},
                             error_y={"type": "data", "array": 1.96 * arm["se"]},
                             showlegend=col == 1), row=1, col=col)
    truth = [-h.intent_to_treat_contrast(float(d), stratum) for d in grid]
    fig.add_trace(go.Scatter(x=grid, y=truth, mode="lines", name="truth",
                             line={"color": "#111", "width": 1.6, "dash": "dot"},
                             showlegend=col == 1), row=1, col=col)
    fig.add_hline(y=0, line={"color": "rgba(0,0,0,0.3)", "dash": "dot"}, row=1, col=col)
    fig.update_xaxes(title_text=f"dose ({h.DOSE_UNIT})", row=1, col=col, gridcolor=h.GRID)
fig.update_yaxes(title_text=f"reduction in SBP ({h.OUTCOME_UNIT})", row=1, col=1, gridcolor=h.GRID)
fig.update_layout(height=430, template="plotly_white",
                  title="A Hill curve per stratum — and where the family runs out",
                  legend={"orientation": "h", "y": 1.12, "x": 0.0},
                  margin={"l": 60, "r": 30, "t": 100, "b": 50})
fig

## 3. The negative result

In the two younger strata the fitted curve tracks the arm means and the truth. In the
oldest one it cannot: the arm means go up, up, and then sharply down, and no member of
the Hill family does that. The fit compromises — it flattens the curve and inflates the
residual scale — and the compromise is visible as a residual that is large and
*systematic in dose*, not as a warning.

In [ ]:
rows = []
for stratum in h.STRATA:
    result = fits[stratum]
    theta = {name: float(result.posterior.summary(name).mean)
             for name in ("alpha", "beta_dose", "k_dose", "s_dose")}
    arm = observed[observed["stratum"] == stratum]
    predicted = forward(result.surface, {"dose": arm["dose"].to_numpy()}, theta)
    for dose, seen, fitted in zip(arm["dose"], arm["mean"], np.ravel(predicted), strict=True):
        rows.append({"stratum": h.STRATUM_LABEL[stratum], "dose": float(dose),
                     "observed": float(seen), "fitted": float(fitted),
                     "residual": float(seen - fitted)})
residuals = pd.DataFrame(rows)
print(residuals.round(2).to_string(index=False))
print("\nlargest arm-mean residual by stratum:")
print(residuals.groupby("stratum")["residual"].apply(lambda r: r.abs().max()).round(2).to_string())
print("\nresidual scale sigma by stratum:")
print({h.STRATUM_LABEL[s]: round(float(fits[s].posterior.summary("sigma").mean), 2)
       for s in h.STRATA})

In [ ]:
fig = h.figure("Arm-mean residual against dose: the misfit is systematic, not noise",
               f"dose ({h.DOSE_UNIT})", f"observed − fitted ({h.OUTCOME_UNIT})", height=380)
for stratum in h.STRATA:
    rows = residuals[residuals["stratum"] == h.STRATUM_LABEL[stratum]]
    fig.add_trace(go.Scatter(x=rows["dose"], y=rows["residual"], mode="lines+markers",
                             name=h.STRATUM_LABEL[stratum], line={"color": h.STRATUM_COLOR[stratum],
                                                                  "width": 2.4},
                             marker={"size": 9}))
fig.add_hline(y=0, line={"color": "rgba(0,0,0,0.45)"})
fig

This is the right failure mode. A monotone family asked to describe a reversal should
produce a bad fit rather than a smooth lie, and `sigma` should absorb the difference —
which is exactly what it did: 7.4 mmHg in the oldest band against 6.2 in the other two.

## 3b. The families that can describe it

`axiom.surface` also ships three **basis** families, whose response is a signed sum over
a fixed basis rather than one amplitude times a monotone shape. They are what to reach
for when the curve might turn over:

- `PolynomialKernel(degree=d)` — `u, u², …, u^d` in `u = dose / reference_dose`;
- `SplineKernel(knots=…)` — a natural cubic spline, cubic between the boundary knots and
  **linear outside** them, so the extrapolation is a line rather than a runaway cubic;
- `PiecewiseLinearKernel(knots=…)` — a broken stick whose coefficients read as slope
  changes at stated doses.

Every coefficient is signed and the response is linear in all of them, so
`parameter_roles` reports them all as `linear` and `surface.linearize` is exact rather
than a local approximation.

And one that commits to less again: `GaussianProcessKernel` puts a stationary **GP
prior** on the dose–response and estimates its smoothness, rather than being handed a
knot set. It is the Hilbert-space basis approximation, so structurally it is one more
basis — with the coefficient scales tied together by a lengthscale instead of free.
Its accuracy is a number you can check: `covariance_error` compares the covariance the
basis implies against the exact one, and the fit below reports whether the lengthscale
it settled on is one the basis can actually represent.

In [ ]:
CANDIDATES: dict[str, AnyKernel] = {
    "hill": spec.kernels["dose"],
    "polynomial(3)": PolynomialKernel(reference_dose=40.0, amplitude_scale=10.0, degree=3),
    "spline(3 knots)": SplineKernel(reference_dose=40.0, amplitude_scale=10.0,
                                    knots=(10.0, 20.0, 30.0)),
    "piecewise_linear": PiecewiseLinearKernel(reference_dose=40.0, amplitude_scale=10.0,
                                              knots=(15.0, 30.0)),
    "gaussian_process": GaussianProcessKernel(reference_dose=48.0, amplitude_scale=10.0),
}
arm_doses = np.asarray(sorted(h.DOSE.values()))
refits: dict[str, FitResult] = {}
print(f"oldest stratum: arm means "
      f"{np.round(observed[observed['stratum'] == 'age_51_plus'].sort_values('dose')['mean'].to_numpy(), 2)}")
print(f"\n{'family':18s} {'sigma':>6} {'max|resid|':>11}  parameter roles")
for label, kern in CANDIDATES.items():
    candidate = spec.model_copy(update={"kernels": {"dose": kern}})
    result = fit(candidate, stratum_panel("age_51_plus"), backend="laplace", draws=800,
                 chains=1, seed=SEED)
    refits[label] = result
    theta_c = {p.name: float(result.posterior.summary(p.name).mean)
               for p in result.surface.model.parameters if p.name != "sigma"}
    fitted = np.ravel(forward(result.surface, {"dose": arm_doses}, theta_c))
    arm = observed[observed["stratum"] == "age_51_plus"].sort_values("dose")["mean"].to_numpy()
    roles = {k: v for k, v in parameter_roles(candidate).items() if k not in ("sigma", "alpha")}
    note = ""
    if label == "gaussian_process":
        ell = float(result.posterior.summary("ell_dose").mean)
        note = f"  ell={ell:.2f} representable={kern.sufficient_for(ell)}"
    print(f"{label:18s} {result.posterior.summary('sigma').mean:6.2f} "
          f"{np.max(np.abs(arm - fitted)):11.2f}  {sorted(set(roles.values()))}{note}")

In [ ]:
fine = np.linspace(0.0, 48.0, 97)
fig = h.figure("The oldest stratum, fitted by four families",
               f"dose ({h.DOSE_UNIT})", f"reduction in SBP ({h.OUTCOME_UNIT})", height=430)
palette = ("#5b6472", "#2f7fd1", "#8a63c4", "#3aa17e", "#c9a227")
for (label, result), colour in zip(refits.items(), palette):
    theta_c = {p.name: float(result.posterior.summary(p.name).mean)
               for p in result.surface.model.parameters if p.name != "sigma"}
    fig.add_trace(go.Scatter(x=fine, y=np.ravel(forward(result.surface, {"dose": fine}, theta_c)),
                             mode="lines", name=label, line={"color": colour, "width": 2.4}))
arm = observed[observed["stratum"] == "age_51_plus"].sort_values("dose")
fig.add_trace(go.Scatter(x=arm["dose"], y=arm["mean"], mode="markers", name="arm mean",
                         marker={"color": "#111", "size": 11},
                         error_y={"type": "data", "array": 1.96 * arm["se"]}))
fig.add_trace(go.Scatter(x=fine, y=[-h.intent_to_treat_contrast(float(d), "age_51_plus") for d in fine],
                         mode="lines", name="truth",
                         line={"color": "#111", "width": 1.6, "dash": "dot"}))
fig.add_hline(y=0, line={"color": "rgba(0,0,0,0.35)", "dash": "dot"})
fig

The monotone family flattens; the basis families follow the turn. That does not make
them the right choice everywhere — in the two younger strata, where the response really
is monotone, the Hill curve fits as well on fewer parameters and its `k` and `beta` mean
something a pharmacologist can argue with, whereas a spline coefficient does not. The
rule this notebook ends up with is the ordinary one: use the family whose *shape
assumption* you are willing to defend, and let a fit that cannot hold the shape say so
out loud, as this one did.

## 4. What four dose levels can and cannot identify

Even where the family fits, the design is thin. `identifiability_ridge` computes the
Fisher information at the design actually run and reports the parameter combination the
data barely constrain. Four dose levels are not enough to separate *how high the curve
goes* from *how steeply it gets there*: you can raise the ceiling and flatten the rise
and pass through the same four arm means.

In [ ]:
young = fits["age_25_35"]
theta = {name: float(young.posterior.summary(name).mean)
         for name in ("alpha", "beta_dose", "k_dose", "s_dose")}
noise = float(young.posterior.summary("sigma").mean)
run_doses = {"dose": np.repeat(np.array(list(h.DOSE.values())), 15)}
information = fisher_information(young.surface, run_doses, theta, noise, method="finite")
assert isinstance(information, FisherInformation)
print("parameters      :", information.parameters)
print("observations    :", information.n_observations)
print("determinant     :", f"{information.det:.4g}", "| singular:", information.singular)
print("min eigenvalue  :", f"{information.min_eigenvalue:.4g}")

ridge = identifiability_ridge(young.surface, run_doses, theta, noise,
                              pairs=(("beta_dose", "k_dose"), ("beta_dose", "s_dose"),
                                     ("k_dose", "s_dose")), method="finite")
assert isinstance(ridge, IdentifiabilityRidge)
print("\ncondition number:", f"{ridge.condition_number:,.1f}",
      f"(min eigenvalue {ridge.min_eigenvalue:.4g}, max {ridge.max_eigenvalue:.4g})")
print("worst-determined direction — the combination the data barely constrain:")
for name, weight in sorted(ridge.direction.items(), key=lambda kv: -abs(kv[1])):
    print(f"   {name:12s} {weight:+.3f}")
print("pairwise correlations at that design:")
for (a, b), corr in zip(ridge.pairs, ridge.correlations, strict=True):
    print(f"   {a:12s} {b:12s} {corr:+.3f}")

In [ ]:
matrix = np.asarray(information.as_array())
scale = np.sqrt(np.diag(matrix))
correlation = matrix / np.outer(scale, scale)
fig = h.figure("Fisher information at the doses actually run, as a correlation", "", "", height=400)
fig.add_trace(go.Heatmap(z=correlation, x=list(information.parameters), y=list(information.parameters),
                         colorscale="RdBu", zmid=0.0, zmin=-1, zmax=1,
                         text=np.round(correlation, 2), texttemplate="%{text}",
                         colorbar={"title": "corr"}))
fig

## 5. Choosing the doses for the next trial

`design_to_identify` takes a set of candidate dose levels and picks the `n` rows —
replicates allowed — that minimize the expected posterior standard deviation of one
named parameter. It is exchange on the Fisher information, so the answer is a *design*,
not advice.

In [ ]:
bounds = Bounds(treatments=("dose",), low=(0.0,), high=(48.0,))
candidates: Design = equal_spacing(bounds, 13)
print("candidate levels:", [round(float(p[0]), 1) for p in candidates.points])
print("(a full factorial on one treatment is the same set:",
      len(full_factorial(bounds, 13).points), "points)")

PRIOR_SDS = {"alpha": 4.0, "beta_dose": 6.0, "k_dose": 12.0, "s_dose": 2.0}
chosen: dict[str, IdentifyingDesign] = {}
for target in ("k_dose", "beta_dose"):
    picked = design_to_identify(young.surface, candidates, theta, noise, target=target, n=60,
                                prior_sds=PRIOR_SDS, seed=SEED)
    assert isinstance(picked, IdentifyingDesign)
    chosen[target] = picked
    levels = pd.Series([round(float(p[0]), 1) for p in picked.design.points]).value_counts().sort_index()
    print(f"\nto pin down {target}: expected posterior sd {picked.expected_sd:.3f} "
          f"(prior {PRIOR_SDS[target]:.1f}), {picked.passes} exchange passes")
    print("   allocation:", {float(k): int(v) for k, v in levels.items()})

In [ ]:
fig = h.figure("Where 60 units should be dosed, depending on what you want to learn",
               f"dose ({h.DOSE_UNIT})", "units allocated", height=380, barmode="group")
for target, colour in (("k_dose", "#2f7fd1"), ("beta_dose", "#8a63c4")):
    counts = pd.Series([round(float(p[0]), 1) for p in chosen[target].design.points]).value_counts()
    fig.add_trace(go.Bar(x=counts.index, y=counts.to_numpy(),
                         name=f"to identify {target}", marker_color=colour))
for dose in list(h.DOSE.values())[1:]:
    fig.add_vline(x=dose, line={"color": "rgba(0,0,0,0.2)", "dash": "dot"})
fig

The two designs are different, and that is the point: identifying *how far the curve
can go* wants mass at zero and at the top of the range; identifying *where it gets
halfway* wants mass around the bend. HYPER-3 tested 0/10/20/40 because it had to answer
both at once, and paid for it in both.

`expected_posterior_sd` prices any design directly, so the three can be compared on the
same scale without re-running the exchange — including the honest verdict on how much
is available to win.

In [ ]:
def sd_for(doses: np.ndarray, label: str) -> dict[str, float]:
    info = fisher_information(young.surface, {"dose": doses}, theta, noise, method="finite")
    assert isinstance(info, FisherInformation)
    out = expected_posterior_sd(PRIOR_SDS, info)
    assert not is_failure(out)
    print(f"{label:28s} " + "  ".join(f"{k}={v:6.3f}" for k, v in out.items()))
    return out


print(f"{'design':28s} " + "  ".join(f"{k:>13s}" for k in PRIOR_SDS))
as_run = sd_for(np.repeat(np.array(list(h.DOSE.values())), 15), "HYPER-3 as run (0/10/20/40)")
optimal_k = sd_for(np.asarray([p[0] for p in chosen["k_dose"].design.points]), "chosen for k_dose")
optimal_b = sd_for(np.asarray([p[0] for p in chosen["beta_dose"].design.points]), "chosen for beta_dose")
print(f"\nat the same 60 units, choosing doses for k_dose cuts its posterior sd by "
      f"{1 - optimal_k['k_dose'] / as_run['k_dose']:.0%};")
print(f"choosing them for beta_dose cuts that one by "
      f"{1 - optimal_b['beta_dose'] / as_run['beta_dose']:.0%} — and costs "
      f"{optimal_b['k_dose'] / as_run['k_dose'] - 1:.0%} on k_dose.")
print(f"s_dose barely moves under any of them: prior {PRIOR_SDS['s_dose']:.2f} -> "
      f"{as_run['s_dose']:.2f} / {optimal_k['s_dose']:.2f} / {optimal_b['s_dose']:.2f}.")
print("Sixty units cannot learn the shape of a Hill curve, whatever doses they are given.")

## 6. The local slope, which is what a prescriber titrates on

`marginal` evaluates ∂outcome/∂dose on the fitted surface — mmHg per mg. It is the
number that says whether the next 10 mg buys anything, and it is the sharpest test of
whether the family was the right one, because a monotone family's slope **cannot change
sign** however the data are shaped.

In [ ]:
def slope_of(result: FitResult) -> np.ndarray:
    theta_s = {p.name: float(result.posterior.summary(p.name).mean)
               for p in result.surface.model.parameters if p.name != "sigma"}
    return np.ravel(marginal(result.surface, theta_s, {"dose": grid}, "dose"))


fig = h.figure("Marginal effect of the next milligram, oldest stratum",
               f"dose ({h.DOSE_UNIT})", f"{h.OUTCOME_UNIT} of reduction per {h.DOSE_UNIT}",
               height=400)
truth_slope = np.gradient(
    [-h.intent_to_treat_contrast(float(d), "age_51_plus") for d in grid], grid)
fig.add_trace(go.Scatter(x=grid, y=truth_slope, mode="lines", name="truth",
                         line={"color": "#111", "width": 2.0, "dash": "dot"}))
for (label, result), colour in zip(refits.items(), palette):
    fig.add_trace(go.Scatter(x=grid, y=slope_of(result), mode="lines", name=label,
                             line={"color": colour, "width": 2.4}))
fig.add_hline(y=0, line={"color": "rgba(0,0,0,0.45)"})
fig

In [ ]:
crossing = float(grid[int(np.argmax(truth_slope < 0))])
print(f"the true marginal effect turns negative at {crossing:.0f} {h.DOSE_UNIT}")
print(f"\n{'family':18s} {'slope at 40 mg':>15} {'sign change found at':>22}")
print(f"{'truth':18s} {truth_slope[np.argmin(abs(grid - 40))]:15.3f} {crossing:>19.0f} mg")
for label, result in refits.items():
    s_hat = slope_of(result)
    negative = np.flatnonzero(s_hat < 0)
    where = f"{grid[negative[0]]:.0f} mg" if negative.size else "never (monotone)"
    print(f"{label:18s} {s_hat[np.argmin(abs(grid - 40))]:15.3f} {where:>22}")

The truth turns negative at 16 mg. The Hill fit's slope has decayed to **zero** by
40 mg and is never negative anywhere — not because it fits badly there, but because a
monotone family has no way to say the thing that is true. Every flexible family finds
the sign change, within a couple of milligrams, and the GP finds it without being told
where to put a knot.

Two caveats travel with that. The GP's slope at the top of the range is attenuated
(−0.03 against a true −0.84): a stationary GP reverts toward its prior at the edge of
the data, which is the price of not committing to a shape. And a flexible family buys
the sign change with parameters that mean nothing on their own — `beta_dose` and
`k_dose` are quantities a pharmacologist can argue about, `z7_dose` is not.

## 7. The figure that goes in the report

In [ ]:
print("plotly available:", available())
figure = response_curve(fits["age_36_50"], "dose", n_grid=25, mass=0.9)
assert not isinstance(figure, Unsupported)
figure.update_layout(title="axiom.viz.response_curve — 36–50 stratum, 90 % band",
                     height=380, template="plotly_white")
figure

## What this notebook decided

- A Hill surface fitted per stratum recovers the intent-to-treat dose–response in the
  two younger age bands, with an amplitude below the generating Emax because adherence
  is inside the estimand.
- In the oldest band the Hill fit fails, and it fails **informatively**: a saturating
  kernel is monotone, the true curve turns over, and the arm-mean residual is large and
  systematic in dose rather than absent. A basis family — polynomial, natural cubic
  spline, broken stick — or a Gaussian process cuts the worst residual by more than half
  and follows the reversal.
- The decisive number is the **marginal effect**, not the fit. At 40 mg the Hill curve
  reports a positive slope in the band where the drug is raising pressure, because a
  monotone family cannot report anything else. Every flexible family finds the sign
  change; the GP finds it without being handed a knot set, at the price of parameters
  that mean nothing individually.
- With only four dose levels the amplitude and the shape are confounded: their
  correlation at the design run is −0.76, and the shape parameter's expected posterior
  sd is barely below its prior. The trial can say the drug works and roughly how much;
  it cannot say how sharply the curve rises.
- `design_to_identify` picks visibly different dose sets depending on which parameter
  the next trial is for, but at 60 units the gains are modest — roughly a tenth of the
  posterior sd — and buying one parameter costs another. Dose placement is not the
  binding constraint here; sample size is, and `expected_posterior_sd` says so before
  anyone runs the follow-on.
- The marginal effect — mmHg per mg — is the quantity that matters for titration, and
  it is exactly where the monotone family misleads. Notebook 5 stops asking the curve
  and starts monitoring the arms.